# IBM Applied Data Science Capstone — SpaceX Falcon 9
## Predicting first-stage landing success

**Question:** Can historical Falcon 9 launch characteristics help predict first-stage landing success?

This notebook follows the IBM Applied Data Science Capstone workflow: data loading, data quality checks, EDA, SQL, Folium mapping, classification models and evaluation.

**Execution note:** Run every cell from top to bottom. The notebook obtains the course datasets from IBM Skills Network. Generated evidence is saved in `outputs/`.

## 1. Environment and imports

In [ ]:
%pip -q install pandas numpy matplotlib seaborn scikit-learn folium sqlalchemy

In [ ]:
import warnings, json, sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)


## 2. Load the IBM Skills Network course datasets

The curated course dataset for the EDA and machine-learning labs is `dataset_part_2.csv`. A fallback URL is included in case the primary course host changes.

In [ ]:
DATA_URLS = {    "part1": [        "https://raw.githubusercontent.com/danghuydk123-code/IBM-Applied-Data-Science-Capstone-SpaceX/main/data/dataset_part_1.csv",        "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"    ],    "part2": [        "https://raw.githubusercontent.com/danghuydk123-code/IBM-Applied-Data-Science-Capstone-SpaceX/main/data/dataset_part_2.csv",        "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv",        "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/dataset_part_2.csv"    ],    "part3": [        "https://raw.githubusercontent.com/danghuydk123-code/IBM-Applied-Data-Science-Capstone-SpaceX/main/data/dataset_part_3.csv",        "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv"    ],    "geo": [        "https://raw.githubusercontent.com/danghuydk123-code/IBM-Applied-Data-Science-Capstone-SpaceX/main/data/spacex_launch_geo.csv"    ]}def load_csv(urls):    last_error = None    for url in urls:        try:            frame = pd.read_csv(url)            print("Loaded:", url)            return frame        except Exception as exc:            last_error = exc    raise RuntimeError(f"Could not load dataset from any configured URL. Last error: {last_error}")part1 = load_csv(DATA_URLS["part1"])df = load_csv(DATA_URLS["part2"])features = load_csv(DATA_URLS["part3"])geo_df = load_csv(DATA_URLS["geo"])print("Part 1 shape:", part1.shape, "| Part 2 shape:", df.shape, "| Part 3 shape:", features.shape)display(df.head())print(df.info())

## 3. Data understanding and quality checks

In [ ]:
print("Columns:", df.columns.tolist())
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (100 * df.isna().mean()).round(2),
    "unique_values": df.nunique(dropna=True)
})
display(quality)
display(df.describe(include="all").T)
quality.to_csv(OUT / "data_quality_summary.csv")


## 4. Data wrangling and target definition

In [ ]:
required = {"Class", "LaunchSite", "PayloadMass", "Orbit"}
missing_required = required - set(df.columns)
if missing_required:
    raise ValueError(f"Dataset is missing required columns: {missing_required}")

df["Class"] = pd.to_numeric(df["Class"], errors="coerce")
df = df.dropna(subset=["Class"]).copy()
df["Class"] = df["Class"].astype(int)
df = df[df["Class"].isin([0, 1])].copy()

if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

print("Rows retained:", len(df))
print("Landing success rate (%):", round(df["Class"].mean() * 100, 2))
display(df["Class"].value_counts().rename(index={0:"Unsuccessful landing", 1:"Successful landing"}))


## 5. EDA with Matplotlib and Seaborn

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="LaunchSite", order=df["LaunchSite"].value_counts().index)
plt.title("Falcon 9 Launches by Launch Site")
plt.xlabel("Launch site"); plt.ylabel("Number of launches")
plt.xticks(rotation=20, ha="right"); plt.tight_layout()
plt.savefig(OUT / "eda_launches_by_site.png", dpi=180); plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="LaunchSite", hue="Class")
plt.title("First-Stage Landing Outcomes by Launch Site")
plt.xlabel("Launch site"); plt.ylabel("Number of launches")
plt.legend(title="Landing class", labels=["Unsuccessful (0)", "Successful (1)"])
plt.xticks(rotation=20, ha="right"); plt.tight_layout()
plt.savefig(OUT / "eda_outcomes_by_site.png", dpi=180); plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
sns.stripplot(data=df, x="Class", y="PayloadMass", jitter=0.22, alpha=0.65)
plt.title("Payload Mass versus First-Stage Landing Outcome")
plt.xlabel("Landing outcome (0 = unsuccessful, 1 = successful)")
plt.ylabel("Payload mass (kg)")
plt.tight_layout(); plt.savefig(OUT / "eda_payload_vs_outcome.png", dpi=180); plt.show()


In [ ]:
orbit_summary = df.groupby("Orbit", dropna=False)["Class"].agg(
    launches="count", success_rate="mean"
).sort_values("success_rate", ascending=False)
orbit_summary["success_rate_pct"] = orbit_summary["success_rate"] * 100
display(orbit_summary)
plt.figure(figsize=(10, 5))
sns.barplot(data=orbit_summary.reset_index(), x="Orbit", y="success_rate_pct")
plt.title("Landing Success Rate by Orbit")
plt.xlabel("Orbit"); plt.ylabel("Success rate (%)")
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig(OUT / "eda_success_by_orbit.png", dpi=180); plt.show()


In [ ]:
if "Date" in df.columns and df["Date"].notna().any():
    yearly = (df.dropna(subset=["Date"])
                .assign(Year=lambda d: d["Date"].dt.year)
                .groupby("Year")
                .agg(launches=("Class", "size"), success_rate=("Class", "mean")))
    display(yearly)
    fig, ax1 = plt.subplots(figsize=(10, 5))
    yearly["launches"].plot(kind="bar", ax=ax1, color="#286090")
    ax1.set_xlabel("Year"); ax1.set_ylabel("Launch count")
    ax2 = ax1.twinx()
    yearly["success_rate"].plot(ax=ax2, color="#D9534F", marker="o")
    ax2.set_ylabel("Landing success rate")
    plt.title("Launch Volume and Landing Success by Year")
    plt.tight_layout(); plt.savefig(OUT / "eda_yearly_trend.png", dpi=180); plt.show()
else:
    print("Date column unavailable or not parseable; yearly trend skipped.")


## 6. EDA with SQL (SQLite)

In [ ]:
conn = sqlite3.connect(OUT / "spacex_capstone.db")
df.to_sql("launches", conn, if_exists="replace", index=False)

sql_queries = {
    "launches_by_site": """SELECT LaunchSite, COUNT(*) AS launch_count
                           FROM launches GROUP BY LaunchSite
                           ORDER BY launch_count DESC""",
    "success_by_site": """SELECT LaunchSite, COUNT(*) AS total_launches,
                          SUM(Class) AS successful_landings,
                          ROUND(100.0 * AVG(Class), 2) AS success_rate_pct
                          FROM launches GROUP BY LaunchSite
                          ORDER BY success_rate_pct DESC""",
    "payload_by_orbit": """SELECT Orbit, COUNT(*) AS launches,
                           ROUND(AVG(PayloadMass), 2) AS average_payload_kg,
                           ROUND(MAX(PayloadMass), 2) AS maximum_payload_kg
                           FROM launches GROUP BY Orbit
                           ORDER BY launches DESC""",
    "landing_class_counts": """SELECT Class, COUNT(*) AS records
                               FROM launches GROUP BY Class ORDER BY Class"""
}
for query_name, query in sql_queries.items():
    print("\n", query_name.upper())
    result = pd.read_sql_query(query, conn)
    display(result)
    result.to_csv(OUT / f"sql_{query_name}.csv", index=False)
conn.close()


## 7. Folium launch-site map

In [ ]:
geo_cols = {"LaunchSite", "Latitude", "Longitude"}
if geo_cols.issubset(df.columns):
    sites = df[list(geo_cols)].dropna().drop_duplicates("LaunchSite")
    launch_map = folium.Map(
        location=[sites["Latitude"].mean(), sites["Longitude"].mean()],
        zoom_start=4, tiles="CartoDB positron"
    )
    for _, row in sites.iterrows():
        folium.Marker(
            [row["Latitude"], row["Longitude"]],
            tooltip=row["LaunchSite"],
            popup=f'{row["LaunchSite"]}<br>Latitude: {row["Latitude"]:.4f}<br>Longitude: {row["Longitude"]:.4f}',
            icon=folium.Icon(color="blue", icon="rocket", prefix="fa")
        ).add_to(launch_map)
    launch_map.save(str(OUT / "spacex_launch_sites_map.html"))
    display(launch_map)
else:
    print("Latitude/Longitude columns are not in this dataset. Use spacex_launch_geo.csv from the course lab.")


## 8. Prepare data for Plotly Dash

In [ ]:
dashboard_columns = [c for c in [
    "LaunchSite", "PayloadMass", "Class", "BoosterVersion", "Orbit", "FlightNumber"
] if c in df.columns]
dashboard_df = df[dashboard_columns].copy()
dashboard_df.to_csv(OUT / "spacex_launch_dash.csv", index=False)
print("Saved dashboard data:", OUT / "spacex_launch_dash.csv")


## 9. Predictive analysis — classification models

In [ ]:
target = "Class"
candidate_features = [
    "FlightNumber", "PayloadMass", "Flights", "GridFins", "Reused",
    "Legs", "Block", "ReusedCount", "Orbit", "LaunchSite",
    "LandingPad", "Serial"
]
predictors = [c for c in candidate_features if c in df.columns]
X = df[predictors].copy()
y = df[target].copy()

numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Predictors:", predictors)
print("Train size:", X_train.shape, "| Test size:", X_test.shape)


In [ ]:
model_specs = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000),
        {"model__C": [0.01, 0.1, 1, 10]}
    ),
    "SVM": (
        SVC(),
        {"model__kernel": ["linear", "rbf"], "model__C": [0.1, 1, 10]}
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {"model__max_depth": [2, 4, 6, None], "model__min_samples_split": [2, 5, 10]}
    ),
    "KNN": (
        KNeighborsClassifier(),
        {"model__n_neighbors": [3, 5, 7, 9]}
    )
}

model_results = []
best_models = {}
for model_name, (estimator, parameter_grid) in model_specs.items():
    pipeline = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    search = GridSearchCV(pipeline, parameter_grid, cv=5, scoring="accuracy", n_jobs=-1)
    search.fit(X_train, y_train)
    predictions = search.predict(X_test)
    model_results.append({
        "Model": model_name,
        "Best_CV_Accuracy": search.best_score_,
        "Test_Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "Best_Params": str(search.best_params_)
    })
    best_models[model_name] = search.best_estimator_

model_comparison = pd.DataFrame(model_results).sort_values(
    "Best_CV_Accuracy", ascending=False
)
display(model_comparison)
model_comparison.to_csv(OUT / "model_comparison.csv", index=False)


In [ ]:
best_model_name = model_comparison.iloc[0]["Model"]
best_model = best_models[best_model_name]
y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print("Selected model:", best_model_name)
print(classification_report(y_test, y_pred, zero_division=0))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Unsuccessful landing", "Successful landing"]
).plot(cmap="Blues", values_format="d")
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.savefig(OUT / "confusion_matrix.png", dpi=180)
plt.show()


## 10. Export verified results

In [ ]:
verified_results = {
    "records": int(len(df)),
    "success_rate_pct": round(float(y.mean() * 100), 2),
    "selected_model": str(best_model_name),
    "best_cv_accuracy": round(float(model_comparison.iloc[0]["Best_CV_Accuracy"]), 4),
    "test_accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
    "precision": round(float(precision_score(y_test, y_pred, zero_division=0)), 4),
    "recall": round(float(recall_score(y_test, y_pred, zero_division=0)), 4),
    "f1": round(float(f1_score(y_test, y_pred, zero_division=0)), 4),
    "confusion_matrix_class_0_then_1": cm.tolist()
}
with open(OUT / "verified_results.json", "w") as file:
    json.dump(verified_results, file, indent=2)
print(json.dumps(verified_results, indent=2))
print("All outputs saved to:", OUT.resolve())


## 11. Submission evidence checklist

- [ ] Run the notebook from top to bottom and resolve any errors.
- [ ] Verify that every plotted chart reflects the loaded IBM course dataset.
- [ ] Include SQL query results and interpret them in the presentation.
- [ ] Open and inspect `outputs/spacex_launch_sites_map.html`.
- [ ] Run the Dash dashboard and capture real screenshots of its charts.
- [ ] Use the actual contents of `outputs/model_comparison.csv`, `outputs/verified_results.json` and `outputs/confusion_matrix.png`.
- [ ] Add the real GitHub repository URL to the presentation and README.
- [ ] Discuss limitations: sample size, historical coverage, class balance, changes in booster technology and uncertainty in outcome labels.

**Do not substitute external benchmark metrics for results generated by your own executed notebook.**